# Lesson 4: Shared PostgreSQL Team Mode

## Objective

Understand how multiple machines can run local Geond processes while sharing one PostgreSQL-compatible database.

## Prerequisites

- Complete Lesson 1 locally first.
- Read `docs/azure_validation/team_collab_validation.md` before provisioning cloud resources.
- Have a private channel for database passwords if you run the optional team validation.

## Safety

This notebook does not provision Azure by default. Treat shared database URLs as secrets and never commit `connection.local.ps1`, SQL dumps, or live passwords.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path.cwd()
RUN_SHARED_DB_CHECKS = False


def run(args, check=True, env=None):
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=REPO, text=True, capture_output=True, env=env)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result

## Run: confirm local-first baseline

Expected outcome: Geond can run locally before you switch any profile to shared storage.


In [ ]:
run(["uv", "run", "geond", "doctor", "--format", "text"])
run(["uv", "run", "geond", "seed-sample"])

## Configure: optional shared PostgreSQL profile

Expected outcome: the profile shape is clear without exposing a real credential.

```bash
GEOND_DATABASE_PROFILE=azure
AZURE_GEOND_DATABASE_URL='postgresql://<user>:<password>@<host>:5432/geond?sslmode=require'
```


In [ ]:
shared_env = os.environ.copy()
shared_env["GEOND_DATABASE_PROFILE"] = "azure"
shared_env["AZURE_GEOND_DATABASE_URL"] = "<set privately before running>"
print("Profile prepared, but not used until RUN_SHARED_DB_CHECKS=True")

## Run: optional shared database reads

Expected outcome: if you opt in and provide a real shared DB URL through your shell, another machine can see sessions, reservations, handoffs, and dashboard events from the same database.


In [ ]:
if RUN_SHARED_DB_CHECKS:
    env = os.environ.copy()
    env["GEOND_DATABASE_PROFILE"] = "azure"
    run(["uv", "run", "geond", "doctor", "--format", "text"], env=env)
    run(
        ["uv", "run", "geond", "dashboard-overview", "file:///sample/geond", "--limit", "20"],
        env=env,
    )
else:
    print(
        "Shared DB checks skipped. Set RUN_SHARED_DB_CHECKS=True only after "
        "exporting a private shared DB URL."
    )

## Team workflow

Expected outcome: each machine runs local `geond-mcp`, CLI, and dashboard processes; the shared component is PostgreSQL-compatible storage.

- Machine A records sessions, reservations, and handoffs.
- Machine B reads the same workspace through its local MCP server and dashboard.
- Dashboard source metadata shows local, Azure, or remote PostgreSQL without showing passwords.

## Cleanup

For temporary Azure validation, delete the resource group and save sanitized cleanup evidence:

```bash
az group delete --name rg-geond-team-validate-<run-id> --yes
```
